# DataArt Vacancies Scraper — Armenia Jobs

**Source:** [dataart.team/vacancies](https://www.dataart.team/vacancies)  
**Company:** DataArt — global software engineering company, Yerevan R&D office  
**Platform:** Custom React SPA (no ATS)  
**Purpose:** Collect all Yerevan/Armenia IT job postings for thesis NLP analysis.

### Strategy
1. Fetch listing page with `requests.get()` — parse `window.INITIAL_STATE` JSON embedded in HTML
2. Filter vacancies with `"Yerevan"` in `locationTags` → extract slugs
3. For each slug, render detail page with **Playwright** (content requires JS execution)
4. Extract full job text (requirements, responsibilities, tech stack) from rendered DOM
5. Save raw + standardized CSVs

### Why Playwright?
The vacancy listing metadata lives in `window.INITIAL_STATE` (server-rendered JSON),  
but the detailed job description body is React-rendered client-side — not in the raw HTML.

### Ethics & robots.txt
- DataArt vacancies page is public — no login required
- Rate-limited: 2 s between requests
- User-Agent identifies academic research purpose

In [1]:
import json
import re
import time
import asyncio
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pandas as pd
from pathlib import Path
from datetime import date

BASE_URL  = "https://www.dataart.team"
HEADERS   = {
    "User-Agent": "Mozilla/5.0 (compatible; ThesisResearch/1.0; Armenian IT curriculum alignment; academic use)",
    "Accept-Language": "en-US,en;q=0.9",
}
DELAY_S   = 2.0
TODAY     = date.today().isoformat()

BASE_DIR  = Path.cwd()
RAW_DIR   = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR  = BASE_DIR / "data" / "processed" / "jobs"

print(f"Today : {TODAY}")
print(f"Raw   : {RAW_DIR}")
print(f"Proc  : {PROC_DIR}")

Today : 2026-03-22
Raw   : /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc  : /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [1]:
def extract_initial_state(html: str) -> dict:
    """Extract window.INITIAL_STATE JSON from raw HTML."""
    # Look for: window.INITIAL_STATE={"..."};
    m = re.search(r"window\.INITIAL_STATE\s*=\s*(\{.+?\})\s*;", html, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return {}


def clean_text(text: str) -> str:
    """Normalize whitespace in extracted text."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_date(val: str) -> str:
    """Normalize date strings to YYYY-MM-DD."""
    if not val:
        return ""
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})", val.strip())
    if m:
        return f"{m.group(1)}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    return val.strip()


print("Helpers defined.")

Helpers defined.


## Step 1 — Parse listing page, filter Yerevan vacancies

In [1]:
VACANCIES_URL = f"{BASE_URL}/vacancies"

resp = requests.get(VACANCIES_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
state = extract_initial_state(resp.text)

# All vacancies are in state["error404"]["allhotvacancies"]
all_vacancies = state.get("error404", {}).get("allhotvacancies", [])
print(f"Total vacancies in INITIAL_STATE: {len(all_vacancies)}")

# Filter to those with Yerevan in locationTags
yerevan_jobs = []
for v in all_vacancies:
    locs = [tag.get("title", "") for tag in v.get("locationTags", [])]
    if any("yerevan" in loc.lower() for loc in locs):
        yerevan_jobs.append({
            "title": v.get("title", ""),
            "slug":  v.get("slug", "").upper(),
            "url":   f"{BASE_URL}/vacancies/{v.get('slug', '').lower()}",
            "locations": locs,
        })

print(f"Yerevan vacancies: {len(yerevan_jobs)}")
print()
for j in yerevan_jobs:
    print(f"  [{j['slug']}] {j['title']}")

Total vacancies in INITIAL_STATE: 12
Yerevan vacancies: 5

  [NET00181] .NET Engineer
  [DE00171]  Senior Python Data Engineer (Azure)
  [BSI00047] Senior Power BI Engineer
  [BSA00128] Senior Business Analyst with NetSuite Experience
  [ML00100]  AI & ML Engineer


## Step 2 — Extract metadata from detail page INITIAL_STATE

Each detail page's `window.INITIAL_STATE` contains a JSON-LD `JobPosting` microdata block  
with `datePosted`, `employmentType`, `industry`, and location array. Parse this with `requests`.

In [1]:
meta_records = {}

for job in yerevan_jobs:
    url = job["url"]
    print(f"  Metadata: {url}")
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except Exception as e:
        print(f"    ERROR: {e}")
        time.sleep(DELAY_S)
        continue

    state = extract_initial_state(r.text)
    micro = state.get("microData", [])

    # Find JobPosting schema entry
    jld = next((m for m in micro if isinstance(m, dict) and m.get("@type") == "JobPosting"), {})

    posting_date    = normalize_date(jld.get("datePosted", ""))
    employment_type = jld.get("employmentType", "").replace("_", " ").title()
    industry        = jld.get("industry", "Information Technology")
    job_id          = jld.get("identifier", {}).get("value", job["slug"])

    meta_records[job["slug"]] = {
        "posting_date":     posting_date,
        "employment_type":  employment_type,
        "industries":       industry,
        "job_id":           job_id,
    }
    print(f"    posted={posting_date}  type={employment_type}")
    time.sleep(DELAY_S)

print(f"\nMetadata collected for {len(meta_records)} jobs")

  Metadata: https://www.dataart.team/vacancies/net00181
    posted=2026-03-09  type=Full Time
  Metadata: https://www.dataart.team/vacancies/de00171
    posted=2026-03-05  type=Full Time
  Metadata: https://www.dataart.team/vacancies/bsi00047
    posted=2026-03-10  type=Full Time
  Metadata: https://www.dataart.team/vacancies/bsa00128
    posted=2026-03-12  type=Full Time
  Metadata: https://www.dataart.team/vacancies/ml00100
    posted=2026-03-08  type=Full Time

Metadata collected for 5 jobs


## Step 3 — Render detail pages with Playwright to extract full job descriptions

The vacancy body (requirements, responsibilities, tech stack) is rendered client-side by React.  
Playwright launches headless Chromium, waits for the content to hydrate, then extracts all text.

In [1]:
async def scrape_vacancy_detail(browser, url: str) -> str:
    """Render a DataArt vacancy detail page and return the full job text."""
    page = await browser.new_page()
    try:
        await page.goto(url, wait_until="networkidle", timeout=30000)
        # Wait for job title to confirm React has rendered
        await page.wait_for_selector("h1", timeout=15000)
        # DataArt uses a vacancy content wrapper — try common selectors
        selectors = [
            "[class*='VacancyDetail']",
            "[class*='Vacancy_']",
            "[class*='vacancy-content']",
            "main",
        ]
        content_el = None
        for sel in selectors:
            try:
                content_el = await page.query_selector(sel)
                if content_el:
                    break
            except Exception:
                continue
        if content_el:
            text = await content_el.inner_text()
        else:
            # Fallback: grab all body text and strip nav/footer noise
            text = await page.inner_text("body")
        return clean_text(text)
    except Exception as e:
        print(f"    Playwright ERROR {url}: {e}")
        return ""
    finally:
        await page.close()


async def run_playwright_scrape(jobs):
    full_texts = {}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        for job in jobs:
            print(f"  Playwright: [{job['slug']}] {job['title']}")
            text = await scrape_vacancy_detail(browser, job["url"])
            full_texts[job["slug"]] = text
            print(f"    full_text: {len(text)} chars")
            await asyncio.sleep(DELAY_S)
        await browser.close()
    return full_texts


full_texts = asyncio.run(run_playwright_scrape(yerevan_jobs))
print(f"\nFull texts collected: {len(full_texts)}")

  Playwright: [NET00181] .NET Engineer
    full_text: 5342 chars
  Playwright: [DE00171] Senior Python Data Engineer (Azure)
    full_text: 4891 chars
  Playwright: [BSI00047] Senior Power BI Engineer
    full_text: 4673 chars
  Playwright: [BSA00128] Senior Business Analyst with NetSuite Experience
    full_text: 5107 chars
  Playwright: [ML00100] AI & ML Engineer
    full_text: 4958 chars

Full texts collected: 5


## Step 4 — Assemble final records

In [1]:
records = []

for job in yerevan_jobs:
    slug = job["slug"]
    meta = meta_records.get(slug, {})
    title_lower = job["title"].lower()

    # Seniority from title
    if "senior" in title_lower or "lead" in title_lower or "architect" in title_lower:
        seniority = "Senior"
    elif "junior" in title_lower:
        seniority = "Junior"
    elif "intern" in title_lower:
        seniority = "Internship"
    else:
        seniority = "Mid"

    records.append({
        "source":           "dataart",
        "source_url":       job["url"],
        "job_title":        job["title"],
        "company_name":     "DataArt",
        "location":         "Yerevan, Armenia",
        "employment_type":  meta.get("employment_type", "Full Time"),
        "seniority_level":  seniority,
        "industries":       meta.get("industries", "Information Technology"),
        "posting_date":     meta.get("posting_date", ""),
        "deadline":         "",
        "skills_tags":      "",
        "full_text":        full_texts.get(slug, ""),
    })

df = pd.DataFrame(records)
print(f"Records assembled: {len(df)}")
df[["job_title", "seniority_level", "posting_date"]].to_string(index=False) and None
for _, r in df.iterrows():
    print(f"  {r['job_title']} [{r['seniority_level']}] posted={r['posting_date']}")

Records assembled: 5
  .NET Engineer [Mid] posted=2026-03-09
  Senior Python Data Engineer (Azure) [Senior] posted=2026-03-05
  Senior Power BI Engineer [Senior] posted=2026-03-10
  Senior Business Analyst with NetSuite Experience [Senior] posted=2026-03-12
  AI & ML Engineer [Mid] posted=2026-03-08


## Step 5 — Save raw and standardized CSVs

In [1]:
STD_COLS = [
    "source", "source_url", "job_title", "company_name", "location",
    "employment_type", "seniority_level", "industries",
    "posting_date", "deadline", "skills_tags", "full_text",
]

# Raw
raw_path = RAW_DIR / "dataart_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved : {raw_path}  ({len(df)} rows)")

# Standardized
std_df   = df[STD_COLS]
std_path = PROC_DIR / "dataart_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path}  ({len(std_df)} rows)")

Raw saved : /Users/lianaaghamalyan/thesis_data/data/raw/jobs/dataart_jobs_raw.csv  (5 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/dataart_jobs_standardized.csv  (5 rows)


In [1]:
print("=== FIELD COVERAGE ===")
for col in STD_COLS:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    pct = filled / len(std_df) * 100
    print(f"  {col:20s}: {filled}/{len(std_df)} ({pct:.0f}%)")

print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")

=== FIELD COVERAGE ===
  source              : 5/5 (100%)
  source_url          : 5/5 (100%)
  job_title           : 5/5 (100%)
  company_name        : 5/5 (100%)
  location            : 5/5 (100%)
  employment_type     : 5/5 (100%)
  seniority_level     : 5/5 (100%)
  industries          : 5/5 (100%)
  posting_date        : 5/5 (100%)
  deadline            : 0/5 (0%)
  skills_tags         : 0/5 (0%)
  full_text           : 5/5 (100%)

full_text — Min: 4673  Median: 4958  Max: 5342
